# 1. Introduction
*   **Weights & Biases Link:** TODO [Insert the link to your W&B report/view here] [1].
*   **Tools & Sources Used:**
    - NotebookLM: Summarizing course materials and project description, Creating Template of Python Notebook
    - Gemini: Coding support / Learning help
    - Claude: Coding support / Coding Coach


# 2. Setup
*State and justify your setup decisions here [1].*

In [5]:
!pip install datasets gensim wandb nltk fasttext

  Using cached fasttext-0.9.3-cp311-cp311-macosx_14_0_universal2.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [fasttext]


In [1]:
# Import all necessary libraries (e.g., PyTorch, Lightning, Hugging Face Datasets, Weights & Biases) [2, 4, 5]
import torch
from datasets import load_dataset
import wandb
from gensim.models import FastText # https://radimrehurek.com/gensim/models/fasttext.html # https://medium.com/@abhishekjainindore24/fasttext-embedding-technique-9b5d1fba4b50 
import gensim.downloader as api
from nltk.tokenize import word_tokenize
import nltk
import numpy as np
import random
import fasttext

In [2]:
# Reproducibility with seed 42
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
# Login to W&B
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/janis.kneubuehler/.netrc.
wandb: Currently logged in as: janis-kneubuehler (janis-kneubuehler-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# 3. Preprocessing & Data Loading
**Decisions:**
- Tokenization: I use nltk.word_tokenize as recommended in the project discussion and demonstrated in the exercise of SW02.
- Lowercase: I convert to lowercase to reduce the vocabulary size, therefore we have less unknown words
- Removal of unkown/other words:
- Format cleaning:
- Truncation: No truncation made due to 
- Feature selection:
- Input format: 
- Label format: 
- Batching, padding:

In [4]:
# Load the PIQA dataset from revision because dataset scripts are no longer supported
# Use splits like specified in Course Project Slides
train = load_dataset("ybisk/piqa", split="train[:-1000]", revision="refs/convert/parquet")
valid = load_dataset("ybisk/piqa", split="train[-1000:]", revision="refs/convert/parquet")
test = load_dataset("ybisk/piqa", split="validation", revision="refs/convert/parquet")

In [5]:
# Exploring the data
print(f"training: {len(train)}, validation: {len(valid)}, test: {len(test)}")

print("Exploring some random rows in dataset:")
print(train[0])
print(train[12])
print(train[123])
print(train[4563])
print(train[13245]) 

training: 15113, validation: 1000, test: 1838
Exploring some random rows in dataset:
{'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
{'goal': 'how do you open a capri-sun', 'sol1': 'open the straw attached to the juice, and then stick it in the small hole at the front of the pouch.', 'sol2': 'cut the top off with scissors', 'label': 0}
{'goal': 'How do you open a gift without ripping the paper?', 'sol1': 'Hold gift over boiling water to let the steam release the tape adhesive.', 'sol2': 'Hold gift over cold water to let the steam release the tape adhesive.', 'label': 0}
{'goal': 'how to make fancy piped flowers', 'sol1': 'use molded ganache', 'sol2': 'use Russian piping tips', 'label': 1}
{'goal': 'How do you keep steel wool and cotton dry?', 'sol1': 'Place the make-shift fire starter inside a film canister.', 'sol2': 'Place the make-shift fire starter inside your pocket.', 'label': 0}


In [ ]:
# Workaround on mac for SSL certificate error (Code by Gemini)
import os
import ssl
import certifi

# 1. Point the notebook to your trusted certificates
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

# 2. Force Python to allow unverified contexts globally (The safety net)
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

print("SSL Bypass active!")

SSL Bypass active!


In [ ]:
# Load word2vec embeddings
wv = api.load('word2vec-google-news-300')
print(f'Vector size: {wv.vector_size}, Vocabulary size: {len(wv.index_to_key)}')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
def preprocess(assertion):
    # tokenize
    assertion = nltk.word_tokenize(assertion)
    # lowercase all words
    assertion = [word.lower() for word in assertion]

    return assertion

max_seq_length = 0

for example in train:   
    assertion1_tokens = preprocess(example['goal'] + " " + example['sol1'])
    assertion2_tokens = preprocess(example['goal'] + " " + example['sol2'])
    
    # check sequence lenght because of no truncation
    current_max = max(len(assertion1_tokens), len(assertion2_tokens))
    if current_max > max_seq_length:
        max_seq_length = current_max
        
    # Not sure yet: TODO Add new words to our vocabulary

[
  [
    "When boiling butter, when it's ready, you can",
    "To permanently attach metal legs to a chair, you can",
    "how do you indent something?",
    "how do you shake something?",
    "Clean tires",
    ...
    "Soften butter quickly.",
    "How to crack a walnut.",
    "How do you look up words in an Arabic dictionary?",
    "What do you do with bad smelling food?",
    "Make automatic anniversary reminder."
  ],
  [
    "How can I get rid of scratches and  marks on wooden furniture or items?",
    "bake brie",
    "To hold chopsticks",
    "To break the seal on a tight lid of a jar,",
    "how do you move a toy car?",
    ...
    "mug",
    "To spread out spoonfuls of cookie dough.",
    "To keep pantyhose runs from getting worse.",
    "Prevent makeup from running.",
    "pine shavings"
  ],
...,
  [
    "How can you get a free disposable carrying cozy for outdoor picnics?",
    "mouth wash",
    "To help dry out shoes when they are wet from rain,",
    "how to choose a go

# 4. Model Definition
*State and justify your architectural decisions here [1].*

**Architectures to Implement [7, 8]:**
1.  **Architecture 1:** Pre-trained Word Embeddings (choose word2vec, GloVe, or fastText) + a 2-layer classifier with a ReLU non-linearity. Train *only* the classifier.
2.  **Architecture 2:** A 2-layer RNN (use PyTorch's LSTM or GRU) + a 2-layer classifier with a ReLU. Use the exact same word embeddings from Architecture 1 for the input layer, but train the whole network end-to-end.

In [ ]:
# Define Architecture 1 (Embeddings + Classifier)
class EmbeddingClassifier(torch.nn.Module):
    # Your implementation
    pass

# Define Architecture 2 (RNN + Classifier)
class RNNClassifier(torch.nn.Module):
    # Your implementation
    pass


# 5. Training
*State and justify your training decisions here [1].*

*   Discuss your chosen loss function and optimization algorithm.
*   Detail the hyperparameters you are using.


In [ ]:
# Initialize Weights & Biases tracking
# wandb.init(project="piqa-project")

# Training loop for Architecture 1
# Training loop for Architecture 2

# Ensure you are actively tracking experiments, parameters, and code versions so your results are reproducible [9].

# 6. Evaluation & Results
*State and justify your evaluation decisions here [1].*

*   Use the `valid_data` split for hyperparameter tuning and comparing your different models and settings [10].
*   Use the `test_data` split **only** for your final comparison. Do not tune hyperparameters on the test set, as this is considered cheating [10].


In [ ]:
# Add your code to evaluate both architectures on the validation set
# Add your code for the final evaluation on the test set

# 7. Conclusion & Interpretation
*State and justify your conclusions here [1].*

*   Interpret your results. Discuss the specialties of each model type, how they handled the physical commonsense reasoning task, and summarize your findings [1, 9].